# Stage 2: Chart Detection with YOLO

| Version | Date | Author | Description |
| --- | --- | --- | --- |
| 2.0.0 | 2026-01-26 | That Le | Complete Stage 2 documentation |

---

## Overview

Stage 2 detects and crops chart regions from document images using YOLO.

### Input/Output

| Property | Value |
| --- | --- |
| **Input** | `Stage1Output` (List of `CleanImage`) |
| **Output** | `Stage2Output` (List of `DetectedChart`) |
| **Key Libraries** | Ultralytics YOLO |
| **Model** | YOLOv8n or custom trained |

### Processing Flow

```
Stage1Output (Clean Images)
       |
       v
+----------------------+
| 1. Load YOLO Model   | --> yolov8n.pt or custom weights
+----------------------+
       |
       v
+----------------------+
| 2. Run Inference     | --> Detect chart regions
+----------------------+
       |
       v
+----------------------+     - NMS (Non-Max Suppression)
| 3. Filter Detections | --> - Confidence threshold
+----------------------+     - Max detections limit
       |
       v
+----------------------+
| 4. Crop Regions      | --> Save cropped images
+----------------------+
       |
       v
Stage2Output
```

---

## Output Schema

```python
class DetectedChart(BaseModel):
    """Single detected chart from Stage 2."""
    chart_id: str           # Unique identifier (e.g., "chart_001")
    source_image: Path      # Original image path
    cropped_path: Path      # Path to cropped chart image
    bbox: BoundingBox       # Detection coordinates + confidence
    page_number: int        # Source page number

class Stage2Output(BaseModel):
    """Output from Stage 2: Detection."""
    session: SessionInfo            # Processing session info
    charts: List[DetectedChart]     # Detected charts
    total_detected: int             # Total count
    skipped_low_confidence: int     # Filtered out count
```

### BoundingBox Details

```python
class BoundingBox(BaseModel):
    x_min: int      # Left edge (pixels)
    y_min: int      # Top edge (pixels)
    x_max: int      # Right edge (pixels)
    y_max: int      # Bottom edge (pixels)
    confidence: float  # Detection confidence [0-1]
    
    @property
    def width(self) -> int:
        return self.x_max - self.x_min
    
    @property
    def height(self) -> int:
        return self.y_max - self.y_min
```

---

## Configuration

In [ ]:
# ============================================================================
# EXECUTION CONTROL FLAG
# ============================================================================
# Set to True to execute actual examples (loads YOLO model ~6MB)
# Set to False for documentation-only mode
# ============================================================================

EXECUTE_EXAMPLES = True  # <-- Change to True to run examples

print(f"Execution mode: {'ACTIVE' if EXECUTE_EXAMPLES else 'DOCUMENTATION ONLY'}")

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP (Always runs)
# ============================================================================
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")

# Check for YOLO model files
model_files = [
    PROJECT_ROOT / "yolo26n.pt",      # Custom chart detector
    PROJECT_ROOT / "yolov8n.pt",      # Pretrained fallback
]
print("\nAvailable YOLO models:")
for m in model_files:
    status = "FOUND" if m.exists() else "NOT FOUND"
    print(f"  [{status}] {m.name}")

---

## 1. Load YOLO Model

**Library**: Ultralytics - Modern YOLO implementation with easy API.

### Model Options

| Model | Size | Purpose |
| --- | --- | --- |
| `yolo26n.pt` | Custom | Trained on chart detection |
| `yolov8n.pt` | ~6MB | Pretrained general object detection |
| `yolov8s.pt` | ~22MB | More accurate, slower |

In [ ]:
# ============================================================================
# DEMO: Load YOLO Model
# ============================================================================
if EXECUTE_EXAMPLES:
    from ultralytics import YOLO
    
    # Try custom chart model first, fallback to pretrained
    model_path = PROJECT_ROOT / "yolo26n.pt"
    if not model_path.exists():
        model_path = PROJECT_ROOT / "yolov8n.pt"
    if not model_path.exists():
        model_path = "yolov8n.pt"  # Download from ultralytics
    
    model = YOLO(str(model_path))
    print(f"Loaded model: {model_path}")
    print(f"Classes: {model.names}")
else:
    print("[SKIPPED] Set EXECUTE_EXAMPLES = True to load YOLO")
    print("\nExpected behavior:")
    print("  - Loads YOLO model from weights file")
    print("  - Displays model classes (for custom model)")

---

## 2. Run Detection

### Key Parameters

| Parameter | Default | Description |
| --- | --- | --- |
| `conf` | 0.25 | Confidence threshold |
| `iou` | 0.45 | IoU threshold for NMS |
| `max_det` | 10 | Maximum detections per image |

In [ ]:
# ============================================================================
# FUNCTION: Detect Charts in Image
# ============================================================================

def detect_charts(model, image_path: Path, conf_threshold: float = 0.25) -> list:
    """
    Detect chart regions in an image using YOLO.
    
    Args:
        model: Loaded YOLO model
        image_path: Path to input image
        conf_threshold: Minimum confidence (default 0.25)
        
    Returns:
        List of detections, each with:
        - bbox: (x1, y1, x2, y2)
        - confidence: float
        - class_name: str
        
    Example:
        >>> detections = detect_charts(model, Path("page1.png"))
        >>> for det in detections:
        ...     print(f"Chart at {det['bbox']} conf={det['confidence']:.2f}")
    """
    # Run inference
    results = model.predict(
        str(image_path),
        conf=conf_threshold,
        verbose=False,
    )
    
    detections = []
    for box in results[0].boxes:
        cls_id = int(box.cls[0])
        conf = float(box.conf[0])
        xyxy = box.xyxy[0].cpu().numpy().astype(int)
        
        detections.append({
            'bbox': tuple(xyxy),  # (x1, y1, x2, y2)
            'confidence': conf,
            'class_id': cls_id,
            'class_name': model.names[cls_id],
        })
    
    return detections

print("Function defined: detect_charts()")

In [ ]:
# ============================================================================
# DEMO: Run Detection on Sample Image
# ============================================================================
if EXECUTE_EXAMPLES:
    import cv2
    import matplotlib.pyplot as plt
    
    # Find test image
    data_dir = PROJECT_ROOT / "data"
    image_dirs = [data_dir / "academic_dataset" / "images", data_dir / "samples"]
    
    test_image = None
    for d in image_dirs:
        if d.exists():
            images = list(d.glob("*.png")) + list(d.glob("*.jpg"))
            if images:
                test_image = images[0]
                break
    
    if test_image:
        print(f"Processing: {test_image.name}")
        
        # Load and display
        image = cv2.imread(str(test_image))
        image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        # Run detection
        detections = detect_charts(model, test_image, conf_threshold=0.25)
        
        print(f"\nDetections: {len(detections)}")
        for i, det in enumerate(detections):
            x1, y1, x2, y2 = det['bbox']
            print(f"  [{i}] {det['class_name']}: conf={det['confidence']:.2f}, box={det['bbox']}")
        
        # Visualize
        plt.figure(figsize=(10, 8))
        plt.imshow(image_rgb)
        plt.title(f"Test Image: {test_image.name}")
        plt.axis('off')
        plt.show()
    else:
        print("No test images found")
else:
    print("[SKIPPED] Set EXECUTE_EXAMPLES = True to run detection")
    print("\nExpected output:")
    print("  Detections: 1")
    print("  [0] chart: conf=0.85, box=(50, 100, 450, 400)")

---

## 3. Visualize Detection Results

YOLO provides built-in visualization with `results[0].plot()`.

In [ ]:
# ============================================================================
# DEMO: Visualize Detection Boxes
# ============================================================================
if EXECUTE_EXAMPLES:
    if test_image and len(detections) > 0:
        # Use YOLO's built-in plotting
        results = model.predict(str(test_image), conf=0.25, verbose=False)
        annotated = results[0].plot()
        
        plt.figure(figsize=(12, 10))
        plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
        plt.title("Detection Results (YOLO Visualization)")
        plt.axis('off')
        plt.show()
    else:
        print("No detections to visualize")
else:
    print("[SKIPPED] Set EXECUTE_EXAMPLES = True to visualize")

---

## 4. Crop Detected Regions

After detection, each chart region is cropped and saved for Stage 3.

In [ ]:
# ============================================================================
# FUNCTION: Crop Detected Regions
# ============================================================================

def crop_detections(image, detections: list, padding: int = 10) -> list:
    """
    Crop detected regions from image.
    
    Args:
        image: numpy array (BGR or RGB)
        detections: List from detect_charts()
        padding: Extra pixels around bbox (default 10)
        
    Returns:
        List of cropped images (numpy arrays)
        
    Example:
        >>> crops = crop_detections(image, detections)
        >>> for i, crop in enumerate(crops):
        ...     cv2.imwrite(f"chart_{i}.png", crop)
    """
    h, w = image.shape[:2]
    crops = []
    
    for det in detections:
        x1, y1, x2, y2 = det['bbox']
        
        # Add padding (with bounds checking)
        x1 = max(0, x1 - padding)
        y1 = max(0, y1 - padding)
        x2 = min(w, x2 + padding)
        y2 = min(h, y2 + padding)
        
        crop = image[y1:y2, x1:x2].copy()
        crops.append(crop)
    
    return crops

print("Function defined: crop_detections()")

In [ ]:
# ============================================================================
# DEMO: Crop and Display Detected Charts
# ============================================================================
if EXECUTE_EXAMPLES:
    if test_image and len(detections) > 0:
        crops = crop_detections(image, detections)
        
        print(f"Cropped {len(crops)} chart regions")
        
        # Display each crop
        for i, crop in enumerate(crops):
            crop_rgb = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)
            
            plt.figure(figsize=(8, 6))
            plt.imshow(crop_rgb)
            plt.title(f"Cropped Chart {i} ({crop.shape[1]}x{crop.shape[0]})")
            plt.axis('off')
            plt.show()
    else:
        print("No regions to crop")
else:
    print("[SKIPPED] Set EXECUTE_EXAMPLES = True to crop charts")

---

## 5. Multi-Chart Handling

When multiple charts are detected in one image, each gets a unique ID.

```
+-----------------------------------+
|  Page with 3 Charts               |
|  +-------+  +-------+  +-------+  |
|  |Chart 1|  |Chart 2|  |Chart 3|  |
|  +-------+  +-------+  +-------+  |
+-----------------------------------+
              |
              v
    +-------------------+
    | chart_001.png     |  --> Stage 3
    | chart_002.png     |  --> Stage 3
    | chart_003.png     |  --> Stage 3
    +-------------------+
```

In [ ]:
# ============================================================================
# FUNCTION: Generate Chart IDs
# ============================================================================

def generate_chart_ids(session_id: str, page_num: int, num_charts: int) -> list:
    """
    Generate unique chart IDs for a page.
    
    Format: {session}_{page}_chart{index}
    
    Args:
        session_id: Session identifier (e.g., "abc123")
        page_num: Page number (1-indexed)
        num_charts: Number of charts detected
        
    Returns:
        List of chart IDs
        
    Example:
        >>> ids = generate_chart_ids("sess001", 1, 3)
        >>> print(ids)
        ['sess001_p1_chart001', 'sess001_p1_chart002', 'sess001_p1_chart003']
    """
    return [
        f"{session_id}_p{page_num}_chart{i+1:03d}"
        for i in range(num_charts)
    ]

# Demo
sample_ids = generate_chart_ids("sess001", 1, 3)
print("Sample chart IDs:")
for chart_id in sample_ids:
    print(f"  - {chart_id}")

---

## 6. Complete Stage 2 Implementation

### Usage Example

```python
from src.core_engine.stages.s2_detection import Stage2Detection, DetectionConfig

# Initialize Stage 2
config = DetectionConfig(
    model_path="yolo26n.pt",
    confidence_threshold=0.5,
    max_detections_per_image=10,
)
stage2 = Stage2Detection(config)

# Process Stage 1 output
stage2_output = stage2.process(stage1_output)

# Access results
print(f"Total charts: {stage2_output.total_detected}")
for chart in stage2_output.charts:
    print(f"  - {chart.chart_id}: {chart.bbox.width}x{chart.bbox.height}")
```

In [ ]:
# ============================================================================
# DEMO: Full Stage 2 Processing
# ============================================================================
if EXECUTE_EXAMPLES:
    try:
        from src.core_engine.stages.s2_detection import Stage2Detection, DetectionConfig
        
        config = DetectionConfig(
            model_path=str(PROJECT_ROOT / "yolov8n.pt"),
            confidence_threshold=0.5,
            max_detections_per_image=10,
        )
        stage2 = Stage2Detection(config)
        print("Stage 2 initialized successfully")
        
    except ImportError as e:
        print(f"Stage 2 not fully implemented yet: {e}")
        print("\nUse the individual functions above:")
        print("  - detect_charts()")
        print("  - crop_detections()")
        print("  - generate_chart_ids()")
else:
    print("[SKIPPED] Set EXECUTE_EXAMPLES = True to test Stage 2")

---

## Summary

### Stage 2 Key Functions

| Function | Purpose | Output |
| --- | --- | --- |
| `detect_charts()` | Run YOLO inference | List of detections |
| `crop_detections()` | Crop chart regions | List of images |
| `generate_chart_ids()` | Create unique IDs | List of strings |

### Configuration Options

| Parameter | Default | Description |
| --- | --- | --- |
| `model_path` | yolov8n.pt | YOLO weights file |
| `confidence_threshold` | 0.5 | Min detection confidence |
| `max_detections_per_image` | 10 | Limit per image |
| `iou_threshold` | 0.45 | NMS IoU threshold |

### Error Handling

| Error | Severity | Action |
| --- | --- | --- |
| Model not found | CRITICAL | Abort |
| Model load failure | CRITICAL | Abort |
| No detections | NORMAL | Return empty list |
| Low confidence only | WARNING | Log and skip |

### Performance Notes

| Model | GPU (ms) | CPU (ms) | mAP |
| --- | --- | --- | --- |
| YOLOv8n | ~5 | ~50 | 0.85+ |
| Custom chart | ~5 | ~50 | 0.90+ |

---

**Previous**: [Stage 1 - Ingestion](01_stage1_ingestion.ipynb)  
**Next**: [Stage 3 - Extraction](03_stage3_extraction.ipynb)